# Phần 3 – Câu 1: Ngân hàng nào sinh lời cao nhất, vị trí dẫn đầu có ổn định không?
Nhóm 5 PTDL · Trực quan hóa & Phân tích thống kê

**Dữ liệu:** bảng 1 – `roa`, `roe`, `loi_nhuan_sau_thue`, `tong_tai_san`, `von_chu_so_huu`.

| Bước | Việc trong kế hoạch | Kết quả |
|---|---|---|
| 1 | Thống kê mô tả ROA, ROE theo ngân hàng và theo năm | bảng mô tả → `outputs/tam/thong_ke_mo_ta_p3.csv` |
| 2 | Biểu đồ xu hướng theo quý và theo năm | `p3_duong_*`, `p3_xu_huong_nam_*` |
| 3 | Tính ROA/ROE năm (tổng LNST 4 quý / TB số dư cuối 4 quý) + kiểm tra lại | bảng ROA/ROE năm |
| 4 | Xếp hạng 8 ngân hàng mỗi năm: bảng xếp hạng + bump chart | `outputs/xep_hang_nam.csv`, `p3_bump_*`, `p3_bang_xep_hang_*` |
| 5 | Đánh giá ổn định: số năm đứng đầu, Spearman giữa các năm liền kề | bảng ổn định |
| 6 | Trả lời câu 1 | nháp nhận định |

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from scipy import stats
from src import config as cf
from src.data_loader import doc_tai_chinh_quy
from src.thong_ke import (bang_mo_ta, bang_rong, tinh_chi_so_nam, xep_hang, bang_hang,
                          thong_ke_vi_tri, dan_dau_tung_nam, tuong_quan_hang_lien_ke, kendall_w)
from src.ket_qua import xuat_ket_qua
from src.viz import ve_duong_chi_so, ve_hop_chi_so, ve_xu_huong_nam, ve_bump_xep_hang, ve_bang_xep_hang

b1 = doc_tai_chinh_quy()
print(b1.shape)

LUU_HINH = True
def hien(fig, ten=None, show=True):
    global LUU_HINH
    if show:
        fig.show()
    if LUU_HINH and ten:
        try:
            cf.luu_hinh(fig, ten)
        except Exception as e:
            LUU_HINH = False
            print("Không lưu được PNG (tạm tắt lưu hình):", str(e).splitlines()[0])

(160, 33)


---
## 1. Thống kê mô tả ROA, ROE
Dùng ROA, ROE **quý đã năm hóa** (LNST quý × 4 / bình quân số dư đầu–cuối quý), 20 quan sát mỗi ngân hàng. Năm hóa để quý và năm cùng thang đo (%/năm).

In [2]:
mo_ta_p3 = bang_mo_ta(b1, ["roa", "roe"], cac_cap_do=(None, "ma_nh", "nam", ["ma_nh", "nam"]))
print("Số dòng:", len(mo_ta_p3))
mo_ta_p3.head()

Số dòng: 108


,chi_so,cap_do,so_quan_sat,trung_binh,trung_vi,do_lech_chuan,nho_nhat,lon_nhat,ma_nh,nam,loai_hinh_so_huu,mau_du_lieu,don_vi
0,roa,toan_bo,160,1.806279,1.74675,0.901618,-1.2468,6.4176,NaN,NaN,NaN,quy,%
1,roa,ma_nh,20,2.188155,2.21880,0.412176,1.1283,2.9416,ACB,NaN,NaN,quy,%
2,roa,ma_nh,20,0.934230,0.93905,0.219308,0.5100,1.4356,BID,NaN,NaN,quy,%
3,roa,ma_nh,20,1.117135,1.04235,0.349082,0.6522,1.9278,CTG,NaN,NaN,quy,%
4,roa,ma_nh,20,2.462045,2.39260,0.376850,1.7723,3.0629,MBB,NaN,NaN,quy,%


### 1.1 ROE theo ngân hàng (sắp theo trung bình)

In [3]:
bang_rong(mo_ta_p3, "roe", "ma_nh").sort_values("trung_binh", ascending=False)

,so_quan_sat,trung_binh,trung_vi,do_lech_chuan,nho_nhat,lon_nhat
ma_nh,,,,,,
MBB,20,23.26,22.91,3.48,17.79,29.19
ACB,20,23.07,22.83,4.37,11.98,31.46
VCB,20,20.43,19.01,4.34,15.40,29.83
BID,20,17.88,17.63,4.16,9.95,26.91
CTG,20,17.82,16.85,5.43,9.93,29.21
TCB,20,17.63,16.23,3.95,9.36,23.59
STB,20,14.73,17.75,8.95,-17.96,27.09
VPB,20,14.35,12.62,8.20,4.43,39.27


### 1.2 ROA theo ngân hàng

In [4]:
bang_rong(mo_ta_p3, "roa", "ma_nh").sort_values("trung_binh", ascending=False)

,so_quan_sat,trung_binh,trung_vi,do_lech_chuan,nho_nhat,lon_nhat
ma_nh,,,,,,
TCB,20,2.84,2.54,0.70,1.44,3.98
MBB,20,2.46,2.39,0.38,1.77,3.06
ACB,20,2.19,2.22,0.41,1.13,2.94
VPB,20,2.15,1.87,1.23,0.74,6.42
VCB,20,1.75,1.70,0.29,1.23,2.29
CTG,20,1.12,1.04,0.35,0.65,1.93
STB,20,1.01,1.22,0.64,-1.25,1.98
BID,20,0.93,0.94,0.22,0.51,1.44


### 1.3 ROA, ROE theo năm (cả 8 ngân hàng)

In [5]:
pd.concat({"ROE": bang_rong(mo_ta_p3, "roe", "nam"), "ROA": bang_rong(mo_ta_p3, "roa", "nam")}, axis=1)

ROE                                                      \
       so_quan_sat trung_binh trung_vi do_lech_chuan nho_nhat lon_nhat   
nam                                                                      
2021.0          32      18.69    20.11          6.58     8.03    29.21   
2022.0          32      20.53    19.69          7.33     4.43    39.27   
2023.0          32      18.54    19.44          5.59     6.33    27.35   
2024.0          32      18.37    18.36          4.51     8.89    27.30   
2025.0          32      17.11    17.34          7.49   -17.96    26.91   

               ROA                                                      
       so_quan_sat trung_binh trung_vi do_lech_chuan nho_nhat lon_nhat  
nam                                                                     
2021.0          32       1.85     1.86          1.08     0.51     3.98  
2022.0          32       2.02     2.02          1.25     0.58     6.42  
2023.0          32       1.74     1.53          0.68     0.83     2.87  
2024.0          32       1.77     1.78          0.58     0.82     2.89  
2025.0          32       1.64     1.64          0.75    -1.25     2.66

**Cách đọc:** *độ lệch chuẩn* cho biết chỉ số dao động nhiều hay ít giữa các quý. STB và VPB có độ lệch chuẩn ROE lớn nhất (~8–9 điểm %), tức lợi nhuận thất thường; MBB nhỏ nhất (3,5) – sinh lời đều nhất. Trung vị STB (17,8%) cao hơn hẳn trung bình (14,7%) vì quý lỗ Q4/2025 (−18%) kéo trung bình xuống.

In [7]:
(cf.THU_MUC_KET_QUA / "tam").mkdir(parents=True, exist_ok=True)
xuat_ket_qua(mo_ta_p3, "thong_ke_mo_ta", thu_muc=cf.THU_MUC_KET_QUA / "tam")
(cf.THU_MUC_KET_QUA / "tam" / "thong_ke_mo_ta.csv").replace(cf.THU_MUC_KET_QUA / "tam" / "thong_ke_mo_ta_p3.csv")
print("Đã lưu outputs/tam/thong_ke_mo_ta_p3.csv – Phần 6 sẽ gộp với Phần 4")

Đã lưu outputs/tam/thong_ke_mo_ta_p3.csv – Phần 6 sẽ gộp với Phần 4


---
## 2. Biểu đồ xu hướng
### 2.1 Theo quý (ROA, ROE năm hóa)

In [ ]:
for cs in ["roe", "roa"]:
    hien(ve_duong_chi_so(b1, cs), f"p3_duong_{cs}_quy")
    hien(ve_hop_chi_so(b1, cs), f"p3_hop_{cs}")

---
## 3. Tính ROA/ROE năm
$$\text{ROA năm} = \frac{\text{Tổng LNST 4 quý}}{\text{TB tổng tài sản cuối 4 quý}} \times 100 \qquad \text{ROE năm} = \frac{\text{Tổng LNST 4 quý}}{\text{TB VCSH cuối 4 quý}} \times 100$$

Vì sao không lấy trung bình ROA quý? ROA quý × 4 dễ bị phóng đại khi một quý lãi/lỗ bất thường. Cách tính trên dùng lợi nhuận **thực tế cả năm**, mẫu số lấy số dư cuối 4 quý nên không cần số liệu năm 2020.

In [ ]:
nam_df = tinh_chi_so_nam(b1)
nam_df.pivot(index="ma_nh", columns="nam", values="roe_nam").round(2)

nam,2021,2022,2023,2024,2025
ma_nh,,,,,
VCB,20.64,23.63,21.17,18.25,16.26
BID,12.83,18.87,18.91,18.72,18.44
CTG,15.19,16.34,16.76,18.33,20.90
TCB,21.37,19.17,14.58,15.30,15.39
MBB,22.90,24.74,23.44,21.22,20.80
ACB,23.18,25.53,24.39,21.53,17.35
STB,10.68,13.73,17.97,19.85,9.89
VPB,17.60,16.94,7.47,11.21,15.15


In [ ]:
nam_df.pivot(index="ma_nh", columns="nam", values="roa_nam").round(2)

nam,2021,2022,2023,2024,2025
ma_nh,,,,,
VCB,1.64,1.83,1.86,1.76,1.54
BID,0.65,0.92,1.02,1.01,0.99
CTG,0.98,0.98,1.06,1.15,1.31
TCB,3.54,3.13,2.36,2.35,2.39
MBB,2.41,2.70,2.53,2.27,2.03
ACB,1.99,2.44,2.46,2.14,1.64
STB,0.68,0.89,1.21,1.41,0.71
VPB,2.43,2.82,1.15,1.84,2.15


### 3.1 Xu hướng ROA, ROE năm

In [ ]:
for cs in ["roe_nam", "roa_nam"]:
    hien(ve_xu_huong_nam(nam_df, cs), f"p3_xu_huong_{cs}")

---
## 4. Xếp hạng 8 ngân hàng mỗi năm

In [ ]:
xh_roe = xep_hang(nam_df, "roe_nam")
xh_roa = xep_hang(nam_df, "roa_nam")
xh_roe.head(8)

,chi_so,ma_nh,nam,loai_hinh_so_huu,gia_tri,hang,hang_nam_truoc,thay_doi_hang
0,roe_nam,ACB,2021,Tư nhân,23.181890,1,NaN,NaN
1,roe_nam,MBB,2021,Tư nhân,22.895061,2,NaN,NaN
2,roe_nam,TCB,2021,Tư nhân,21.366931,3,NaN,NaN
3,roe_nam,VCB,2021,Quốc doanh,20.641843,4,NaN,NaN
4,roe_nam,VPB,2021,Tư nhân,17.600240,5,NaN,NaN
5,roe_nam,CTG,2021,Quốc doanh,15.193665,6,NaN,NaN
6,roe_nam,BID,2021,Quốc doanh,12.828717,7,NaN,NaN
7,roe_nam,STB,2021,Tư nhân,10.680773,8,NaN,NaN


In [ ]:
xuat_ket_qua(pd.concat([xh_roe, xh_roa], ignore_index=True), "xep_hang_nam")
print("Đã lưu outputs/xep_hang_nam.csv")

Đã lưu outputs/xep_hang_nam.csv


### 4.1 ROE

In [ ]:
hien(ve_bang_xep_hang(xh_roe), "p3_bang_xep_hang_roe")
hien(ve_bump_xep_hang(xh_roe), "p3_bump_roe")

### 4.2 ROA

In [ ]:
hien(ve_bang_xep_hang(xh_roa), "p3_bang_xep_hang_roa")
hien(ve_bump_xep_hang(xh_roa), "p3_bump_roa")

Bump chart có thể làm nổi bật vài ngân hàng (dùng cho dashboard hoặc slide):

In [ ]:
hien(ve_bump_xep_hang(xh_roe, ma_nh=["ACB", "MBB", "CTG", "STB"]), "p3_bump_roe_noi_bat")

---
## 5. Đánh giá độ ổn định của vị trí dẫn đầu
### 5.1 Số năm đứng đầu, số năm top 3, hạng trung bình

In [ ]:
pd.concat({"ROE năm": thong_ke_vi_tri(xh_roe), "ROA năm": thong_ke_vi_tri(xh_roa)}, axis=1)

ROE năm                                                   \
      so_nam_hang_1 so_nam_top_3 hang_tb hang_tot_nhat hang_kem_nhat   
ma_nh                                                                  
ACB               4            4     1.6             1             4   
CTG               1            1     5.0             1             7   
MBB               0            5     2.0             2             2   
VCB               0            2     4.2             3             6   
BID               0            1     4.6             3             7   
TCB               0            1     5.4             3             7   
STB               0            1     6.4             3             8   
VPB               0            0     6.8             5             8   

                               ROA năm                                     \
      do_lech_chuan_hang so_nam_hang_1 so_nam_top_3 hang_tb hang_tot_nhat   
ma_nh                                                                       
ACB                 1.34             0            2     3.4             2   
CTG                 2.35             0            0     6.4             6   
MBB                 0.00             1            5     2.4             1   
VCB                 1.30             0            0     4.8             4   
BID                 1.52             0            0     7.6             7   
TCB                 1.82             4            5     1.4             1   
STB                 2.30             0            0     6.8             5   
VPB                 1.30             0            3     3.2             2   

                                        
      hang_kem_nhat do_lech_chuan_hang  
ma_nh                                   
ACB               4               0.89  
CTG               7               0.55  
MBB               3               0.89  
VCB               5               0.45  
BID               8               0.55  
TCB               3               0.89  
STB               8               1.30  
VPB               6               1.79

### 5.2 Ai đứng đầu, cách hạng 2 bao xa?
Khoảng cách nhỏ (dưới ~0,5 điểm %) nghĩa là vị trí số 1 dễ thay đổi chỉ vì chênh lệch nhỏ.

In [ ]:
dan_dau_tung_nam(xh_roe)

,nam,hang_1,gia_tri_1,hang_2,gia_tri_2,chenh_lech_diem_pct,trung_binh_8_nh
0,2021,ACB,23.18,MBB,22.90,0.29,18.05
1,2022,ACB,25.53,MBB,24.74,0.79,19.87
2,2023,ACB,24.39,MBB,23.44,0.95,18.09
3,2024,ACB,21.53,MBB,21.22,0.31,18.05
4,2025,CTG,20.90,MBB,20.80,0.10,16.77


In [ ]:
dan_dau_tung_nam(xh_roa)

,nam,hang_1,gia_tri_1,hang_2,gia_tri_2,chenh_lech_diem_pct,trung_binh_8_nh
0,2021,TCB,3.54,VPB,2.43,1.11,1.79
1,2022,TCB,3.13,VPB,2.82,0.31,1.96
2,2023,MBB,2.53,ACB,2.46,0.07,1.70
3,2024,TCB,2.35,MBB,2.27,0.08,1.74
4,2025,TCB,2.39,VPB,2.15,0.24,1.60


### 5.3 Tương quan hạng giữa các năm liền kề
- **Spearman ρ** và **Kendall τ**: so thứ hạng năm này với năm sau. Gần 1 = thứ hạng giữ nguyên; gần 0 = xáo trộn.
- p-value < 0,05: thứ hạng hai năm giống nhau một cách có ý nghĩa (không phải ngẫu nhiên).
- Chỉ có 8 ngân hàng nên ρ cần khoảng ≥ 0,74 mới có ý nghĩa ở mức 5%.

In [ ]:
pd.concat({"ROE năm": tuong_quan_hang_lien_ke(xh_roe).set_index("cap_nam"),
           "ROA năm": tuong_quan_hang_lien_ke(xh_roa).set_index("cap_nam")}, axis=1)

ROE năm                                                \
          so_ngan_hang spearman_rho spearman_p kendall_tau kendall_p   
cap_nam                                                                
2021–2022            8        0.905      0.002       0.786     0.006   
2022–2023            8        0.714      0.047       0.571     0.061   
2023–2024            8        0.833      0.010       0.714     0.014   
2024–2025            8        0.357      0.385       0.286     0.399   

               ROA năm                                                
          so_ngan_hang spearman_rho spearman_p kendall_tau kendall_p  
cap_nam                                                               
2021–2022            8        0.976      0.000       0.929     0.000  
2022–2023            8        0.524      0.183       0.429     0.179  
2023–2024            8        0.857      0.007       0.714     0.014  
2024–2025            8        0.857      0.007       0.714     0.014

### 5.4 Kendall W – mức thống nhất thứ hạng qua cả 5 năm
Một con số tóm tắt: W = 1 nếu 5 năm xếp hạng y hệt, W = 0 nếu hoàn toàn ngẫu nhiên. Tính thêm cho 2021–2024 để xem năm 2025 (STB lỗ quý 4) ảnh hưởng thế nào.

In [ ]:
pd.DataFrame({
    "ROE 2021–2025": kendall_w(xh_roe), "ROE 2021–2024": kendall_w(xh_roe[xh_roe["nam"] <= 2024]),
    "ROA 2021–2025": kendall_w(xh_roa), "ROA 2021–2024": kendall_w(xh_roa[xh_roa["nam"] <= 2024]),
}).T

,kendall_w,chi2,bac_tu_do,p_value,so_nam,so_ngan_hang
ROE 2021–2025,0.589,20.60,7.0,0.0044,5.0,8.0
ROE 2021–2024,0.693,19.42,7.0,0.0070,4.0,8.0
ROA 2021–2025,0.846,29.60,7.0,0.0001,5.0,8.0
ROA 2021–2024,0.839,23.50,7.0,0.0014,4.0,8.0


---
## 6. Trả lời câu 1 (bản nháp – viết lại bằng lời của bạn)

**Ngân hàng nào sinh lời cao nhất?** Câu trả lời phụ thuộc vào thước đo:

- **Theo ROE** (lợi nhuận trên vốn chủ sở hữu – góc nhìn cổ đông): **ACB** dẫn đầu 4/5 năm (2021–2024, ROE 21,5–25,5%), nhưng chỉ hơn hạng 2 từ 0,3 đến 0,9 điểm %. **MBB** đứng hạng 2 **cả 5 năm** (20,8–24,7%) – ngân hàng duy nhất không đổi hạng. Năm 2025, **CTG** vươn lên hạng 1 (20,9%) nhưng chỉ hơn MBB 0,1 điểm %, còn ACB tụt xuống hạng 4.
- **Theo ROA** (lợi nhuận trên tổng tài sản – hiệu quả dùng tài sản): **TCB** dẫn đầu 4/5 năm và luôn nằm trong top 3; MBB đứng đầu năm 2023 (hơn ACB 0,07 điểm %).
- Hai thước đo khác nhau vì ROE = ROA × (tổng tài sản / VCSH). TCB có vốn chủ sở hữu dày (CAR ~15%, cùng VPB cao nhất nhóm) nên ROA cao nhưng ROE chỉ ở mức trung bình; ngược lại các ngân hàng dùng đòn bẩy cao hơn có ROE cao hơn so với ROA.

**Vị trí dẫn đầu có ổn định không?**

- **Khá ổn định ở nhóm đầu, nhưng vị trí số 1 mong manh.** Kendall W = 0,59 với ROE và 0,85 với ROA (đều p < 0,01): thứ hạng các năm thống nhất có ý nghĩa thống kê, ROA ổn định hơn ROE. Tương quan hạng Spearman giữa các năm liền kề với ROE là 0,91 → 0,71 → 0,83 → **0,36** (2024–2025, không có ý nghĩa thống kê); với ROA là 0,98 → 0,52 → 0,86 → 0,86.
- **Năm 2025 xáo trộn mạnh** với ROE: CTG từ hạng 5 lên 1, ACB từ 1 xuống 4, STB từ 3 xuống 8 do lỗ quý 4/2025. Nếu chỉ xét 2021–2024, Kendall W của ROE tăng lên 0,69.
- **Nhóm cuối cũng khá cố định:** theo ROE, VPB và STB xếp cuối phần lớn các năm; theo ROA, BID, STB, CTG luôn ở nửa dưới.

**Hạn chế.** (1) Chỉ 8 ngân hàng, 5 năm nên kiểm định có độ tin cậy thấp; ρ phải ≥ ~0,74 mới có ý nghĩa. (2) Chênh lệch giữa hạng 1 và hạng 2 thường dưới 1 điểm %, nên kết luận "đứng đầu" nhạy với cách tính – nhóm đã đối chiếu với số VNDirect và thấy thứ hạng trùng khớp. (3) Số liệu BCTC là bản sau soát xét/kiểm toán. (4) LNST là lợi nhuận hợp nhất (gồm cả cổ đông thiểu số).